In [ ]:
import sqlite3
import pandas as pd
from google.colab import drive
from google.colab import files
drive.mount('/content/drive')



# Load the uploaded CSV into a pandas dataframe
file_path = "/content/drive/MyDrive/Colab Notebooks/loan_final313.csv"
df = pd.read_csv(file_path)


# Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')
# Create the 'loans_info' table and load the data (Extract Phase)
df.to_sql('loans_info', conn, index=False)


# We calculate the median income first to inject into our SQL
median_income = df['annual_inc'].median()

# This is the exact SQL you will put in your report
sql_query = f"""WITH Stats AS (
    SELECT
        year,
        CAST(COUNT(*) AS FLOAT) AS total_loans_yr,
        SUM(CASE WHEN loan_condition = 'Bad Loan' THEN 1 ELSE 0 END) AS bad_loans_yr
    FROM loans_info
    GROUP BY year
)
SELECT
    l.id,
    l.year,
    l.issue_d,

    CAST(REPLACE(l.term, ' months', '') AS INTEGER) AS term_numeric,
    UPPER(l.home_ownership) AS home_ownership,
    COALESCE(l.annual_inc, {median_income}) AS annual_inc,

    l.loan_amount,
    l.purpose,
    l.loan_condition,

    CAST(REPLACE(l.interest_rate, '%', '') AS FLOAT) AS interest_rate,

    l.grade,
    l.total_pymnt,
    l.recoveries,
    l.region,

    (l.total_pymnt - l.loan_amount) AS profitability,
    CASE WHEN l.loan_condition = 'Bad Loan' THEN 1 ELSE 0 END AS risk_flag,

    (COALESCE(l.annual_inc, {median_income}) / NULLIF(l.loan_amount, 0)) AS income_to_loan_ratio,
    (s.bad_loans_yr / NULLIF(s.total_loans_yr, 0)) AS default_rate_indicator

FROM loans_info l
JOIN Stats s ON l.year = s.year

WHERE l.loan_amount IS NOT NULL
  AND l.interest_rate IS NOT NULL
  AND l.purpose IS NOT NULL;
"""

# Execute the SQL Query and load the results into a new Dataframe (Load Phase)
cleaned_df = pd.read_sql_query(sql_query, conn)


# Save the cleaned data to a new CSV file
cleaned_df.to_csv('cleaned_loan_data.csv', index=False)
print("ETL Complete! Downloading cleaned_loan_data.csv")

# Automatically trigger the download to your computer
files.download('cleaned_loan_data.csv')

Mounted at /content/drive
ETL Complete! Downloading cleaned_loan_data.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

# 1. First 10 Records"
query1 = "SELECT * FROM loans_info LIMIT 10;"
display(pd.read_sql_query(query1, conn))

,id,year,issue_d,final_d,emp_length_int,home_ownership,home_ownership_cat,income_category,annual_inc,income_cat,...,loan_condition_cat,interest_rate,grade,grade_cat,dti,total_pymnt,total_rec_prncp,recoveries,installment,region
0,6297794,2013,01-07-2013,1122015,10.0,MORTGAGE,3,Low,55000,1,...,0,10.64,B,2,20.57,11630.35000,10000.00,0.0,325.69,cannught
1,4526163,2013,01-05-2013,1102014,9.0,RENT,1,Low,60000,1,...,0,21.00,E,5,8.82,13165.83707,10575.00,0.0,398.42,cannught
2,9001808,2013,01-11-2013,1012016,10.0,MORTGAGE,3,Low,63000,1,...,0,13.67,B,2,8.72,7849.66000,6033.91,0.0,301.91,munster
3,7306367,2013,01-09-2013,1122015,10.0,MORTGAGE,3,Low,100000,1,...,0,16.20,C,3,19.52,6924.42000,3707.15,0.0,256.46,ulster
4,6532028,2013,01-08-2013,1082014,10.0,MORTGAGE,3,Low,85000,1,...,0,16.78,C,3,15.49,27523.60099,24000.00,0.0,853.04,ulster
5,6316917,2013,01-07-2013,1102014,1.0,RENT,1,Low,82500,1,...,0,9.71,B,2,7.96,9901.40164,9000.00,0.0,289.19,munster
6,1570001,2012,01-10-2012,1102015,NaN,RENT,1,Low,55000,1,...,0,15.31,C,3,24.34,15040.97044,12000.00,0.0,417.81,Northern-Irl
7,26199528,2014,01-10-2014,1012016,10.0,RENT,1,Low,67980,1,...,0,6.03,A,1,6.54,4761.17000,4158.69,0.0,304.36,Northern-Irl
8,35074071,2014,01-11-2014,1122015,10.0,RENT,1,Low,51000,1,...,0,8.67,B,2,12.40,2876.52000,2321.36,0.0,221.53,Northern-Irl
9,3725985,2013,01-03-2013,1012016,1.0,MORTGAGE,3,Low,40000,1,...,0,16.29,C,3,20.73,14435.85000,8041.99,0.0,437.45,ulster


In [ ]:
# 2. Total Number of Records"
query2 = "SELECT COUNT(*) AS total_records FROM loans_info;"
display(pd.read_sql_query(query2, conn))

,total_records
0,70000


In [ ]:
# 3. Unique Home Ownership Categories
query3 = "SELECT DISTINCT home_ownership FROM loans_info;"
display(pd.read_sql_query(query3, conn))

,home_ownership
0,MORTGAGE
1,RENT
2,OWN
3,OTHER
4,NONE


---